# v2: is the distillation gain the teacher, or regularisation? (one seed per run)

Six conditions on identical data, LoRA config and optimizer, differing only in what is listed:

| condition | what differs from `sft_only` |
|---|---|
| `base` | no training |
| `sft_only` | (reference) hard-label SFT, 3 epochs |
| `sft_early` | stopped after 1 epoch (early-stopping control) |
| `sft_ls` | label smoothing 0.1 (regularisation control) |
| `self_distill` | 0.5 SFT + 0.5 KL, but the teacher is the frozen base student itself: no external knowledge |
| `distilled` | 0.5 SFT + 0.5 KL from Qwen3-14B |
| `distilled_8b` (optional) | same, from Qwen3-8B |

If `self_distill` matches `distilled`, the gain is regularisation; if `distilled` beats it, the teacher contributes information.

**How to run.** Set `SEED` in the first code cell (0, then 1, then 2), Save Version -> Save & Run All. Each seed has its own run tag (`v2-seed<SEED>`) and resumes on its own. Needs the Kaggle secret `HF_TOKEN` (write access to a Hugging Face repo); never paste a token into a cell. Enable GPU T4 x2 and Internet.

Rough time per seed on T4 x2: training ~70 min, evaluation ~3.5 h (221 tasks per condition: 121 synthetic multi-step tasks on unseen tools + 100 held-out single-step tasks on the training tools), so ~5 h. The base model is evaluated only for seed 0 (greedy decoding makes it identical across seeds).

In [ ]:
# ---- edit these two lines, then Save & Run All. One seed per run: 0, then 1, then 2. ----
SEED = 0
INCLUDE_8B = False    # also train/evaluate the Qwen3-8B-teacher condition (~+25 min per seed)

import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['ADBENCH_SEED'] = str(SEED)

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ['ADBENCH_RUN_TAG'] = f'v2-seed{SEED}'   # one run tag per seed
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

In [ ]:
from adbench import pipeline_state as ps

ps.restore()
ps.check_upload()

In [ ]:
CONDITIONS_V2 = ["base", "sft_only", "sft_early", "sft_ls", "self_distill", "distilled"] + (["distilled_8b"] if INCLUDE_8B else [])
TRAINED = [c for c in CONDITIONS_V2 if c != "base"]
EVAL_CONDITIONS = [c for c in CONDITIONS_V2 if c != "base" or SEED == 0]
print("seed", SEED, "| train:", CONDITIONS_V2, "| evaluate:", EVAL_CONDITIONS)

## 1. Data

In [ ]:
def prepare_data():
    run_module("adbench.data.prepare", "--config", "configs/data.yaml")
    run_module("adbench.data.general_eval", "--config", "configs/experiment.yaml")


ps.run_stage("data", prepare_data)

## 2. Training

In [ ]:
for condition in CONDITIONS_V2:
    ps.run_stage(
        f"train_{condition}",
        lambda condition=condition: run_module("adbench.training.train", "--condition", condition),
    )

Guard: stop now if a trained model does not produce clean tool calls, before the long evaluation.

In [ ]:
def guard():
    for condition in TRAINED:
        run_module("adbench.evaluation.diagnose", "--condition", condition,
                   "--skip-training-target", "--n", "5", "--require-tool-call", "--max-failures", "1")


ps.run_stage("guard", guard)

## 3. Evaluation (greedy decoding; unseen-tool and seen-tool sets)

In [ ]:
import json

from adbench.evaluation.run_eval import evaluate_condition, write_eval_outputs
from adbench.training.train import REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")
chain_lengths = experiment_config["harness"]["chain_lengths"]

all_rows = []
perplexities = {}
for condition in EVAL_CONDITIONS:
    stage = f"eval_{condition}"
    cache_file = ps.stages_dir() / f"{stage}.json"
    if ps.is_done(stage) and cache_file.exists():
        saved = json.loads(cache_file.read_text(encoding="utf-8"))
        rows, perplexity = saved["rows"], saved["perplexity"]
        print(f"=== {condition}: restored from an earlier run ===")
    else:
        print(f"=== Evaluating {condition} ===")
        rows, perplexity = evaluate_condition(condition, experiment_config, models_config, chain_lengths)
        for row in rows:
            row["seed"] = SEED
        ps.stages_dir().mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps({"rows": rows, "perplexity": perplexity}), encoding="utf-8")
        ps.finish(stage)
    all_rows.extend(rows)
    perplexities[condition] = perplexity
    print(f"{condition}: {len(rows)} tasks, perplexity={perplexity:.2f}")
    output = write_eval_outputs(all_rows, perplexities, REPO_ROOT / "results")

print("wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json")

In [ ]:
import pandas as pd

df = pd.DataFrame(all_rows)[["condition", "task_set", "chain_length", "success"]]
table = df.groupby(["task_set", "condition", "chain_length"])["success"].mean().unstack("chain_length").round(3)
print(f"seed {SEED}: full-chain success")
table

Seed 0 was first evaluated before per-step argument accuracy was logged, so its seen-tool rows lack it. This stage re-runs only those 100 held-out tasks per condition (about 10 minutes each) and writes `results/seen_args/`. Seeds 1 and 2 log argument accuracy in the main evaluation and skip it. To run it for seed 0, import this notebook again with `SEED = 0`: every finished stage is restored and skipped, and only this one runs.

In [ ]:
if SEED == 0:
    ps.run_stage(
        "seen_args",
        lambda: run_module("adbench.evaluation.run_eval", "--seen-only", "--conditions", ",".join(CONDITIONS_V2),
                           "--output-dir", "results/seen_args"),
    )

In [ ]:
ps.push(f"final results, seed {SEED}")